# 💰 04 — Profitability Analysis
## Customer Profit = Revenue − COGS − Discounts − Returns − Service − Acquisition

Computes true customer-level profitability, not just revenue, to identify which customers actually generate profit.

---

In [1]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath('..'))
import config

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

## 1. Load Data & Run Profitability

In [2]:
from src import profitability

segmented = pd.read_csv(os.path.join(config.DATA_PROCESSED, 'customer_segmented.csv'))
transactions = pd.read_csv(os.path.join(config.DATA_RAW, 'transactions.csv'))
customers = pd.read_csv(os.path.join(config.DATA_RAW, 'customers.csv'))
engagement = pd.read_csv(os.path.join(config.DATA_RAW, 'engagement.csv'))

profitable = profitability.run(
    segmented_customers=segmented,
    transactions=transactions,
    customers=customers,
    engagement=engagement
)


PHASE 4: Customer Profitability
  Computing customer profitability...

  Profitability Summary:
    Total Revenue:     R$   13,022,744
    Total Profit:      R$    2,749,213
    Avg Profit Margin:        1.2%
    Unprofitable:      32,641 customers (34.0%)

  ⚠ High revenue, low margin: 342 customers
  Generating profitability charts...

  Profit by RFM Segment:
        rfm_segment  avg_profit  avg_margin  count
            At Risk   40.972040    0.195956   9125
          Champions   89.459486    0.303209   6050
            Dormant   -8.724711   -0.440667   2440
 High Value At Risk   90.286083    0.308119   7011
    Loyal Customers   51.241084    0.227552  15062
    Needs Attention   24.798112   -0.111816  40951
Potential Loyalists  -40.366938   -0.170454  15457

  ✓ Profitability data → d:\Customer Profitability and Retention Analytics\data\processed\customer_profitable.csv


## 2. Pareto Analysis (80/20 Rule)

In [3]:
# Sort customers by profit and compute cumulative %
sorted_profit = profitable.sort_values('customer_profit', ascending=False)
sorted_profit['cumulative_profit'] = sorted_profit['customer_profit'].cumsum()
total_profit = sorted_profit['customer_profit'].sum()
sorted_profit['cumulative_pct'] = sorted_profit['cumulative_profit'] / total_profit * 100
sorted_profit['customer_pct'] = np.arange(1, len(sorted_profit) + 1) / len(sorted_profit) * 100

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(sorted_profit['customer_pct'], sorted_profit['cumulative_pct'],
        color='#e74c3c', linewidth=2.5)
ax.axhline(80, color='gray', linestyle='--', alpha=0.5, label='80% of profit')
ax.axvline(20, color='gray', linestyle=':', alpha=0.5, label='20% of customers')
ax.fill_between(sorted_profit['customer_pct'], sorted_profit['cumulative_pct'], alpha=0.1, color='#e74c3c')
ax.set_title('Pareto Analysis — Cumulative Profit Contribution', fontsize=14, fontweight='bold')
ax.set_xlabel('% of Customers (ranked by profit)')
ax.set_ylabel('% of Total Profit')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# How much profit do top 20% generate?
top_20_pct = sorted_profit[sorted_profit['customer_pct'] <= 20]['customer_profit'].sum()
print(f"Top 20% of customers generate {top_20_pct/total_profit*100:.1f}% of total profit")

Top 20% of customers generate 102.0% of total profit


C:\Users\HP\AppData\Local\Temp\ipykernel_20780\2454745790.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Value Tier Breakdown

In [4]:
tier_summary = profitable.groupby('value_tier').agg(
    count=('customer_id', 'count'),
    total_profit=('customer_profit', 'sum'),
    avg_profit=('customer_profit', 'mean'),
    avg_margin=('profit_margin', 'mean'),
).round(2)
tier_summary['pct'] = (tier_summary['count'] / tier_summary['count'].sum() * 100).round(1)
print(tier_summary.to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

tier_colors = {'High Value': '#2ecc71', 'Medium Value': '#f39c12', 'Low Value': '#e74c3c'}
for tier in ['High Value', 'Medium Value', 'Low Value']:
    data = profitable[profitable['value_tier'] == tier]['customer_profit']
    axes[0].hist(data.clip(-200, 500), bins=40, alpha=0.6, label=tier, color=tier_colors[tier])

axes[0].set_title('Profit Distribution by Value Tier', fontweight='bold')
axes[0].set_xlabel(f'Customer Profit ({config.CURRENCY_SYMBOL})')
axes[0].legend()

# Profit margin by segment
seg_margin = profitable.groupby('rfm_segment')['profit_margin'].mean().sort_values()
colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in seg_margin.values]
seg_margin.plot(kind='barh', ax=axes[1], color=colors)
axes[1].set_title('Avg Profit Margin by Segment', fontweight='bold')
axes[1].axvline(0, color='black', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

              count  total_profit  avg_profit  avg_margin   pct
value_tier                                                     
High Value    31615    3340393.46      105.66        0.38  33.4
Low Value     32087   -1072430.39      -33.42       -0.58  33.9
Medium Value  30931     481249.88       15.56        0.25  32.7


C:\Users\HP\AppData\Local\Temp\ipykernel_20780\448053054.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
**✅ Profitability analysis complete!** Proceed to `05_churn_prediction.ipynb`.